# Distribution Feeder Modeling

## Creating a FeederModel

Import all required libraries for data profile, connection parameters, database, and feeder:

In [ ]:
import cimgraph.data_profile.rc4_2021 as cim
from cimgraph import ConnectionParameters
from cimgraph.databases.graphdb import GraphDBConnection
from cimgraph.databases.blazegraph import BlazegraphConnection
from cimgraph.models import FeederModel

In [ ]:
# GraphDB Connection
params = ConnectionParameters(url = "http://localhost:7200/repositories/cim_test", 
                              cim_profile='rc4_2021', iec61970_301=8)
graphdb = GraphDBConnection(params)

In [ ]:
# Blazegraph Connection
params = ConnectionParameters(url = "http://localhost:8889/bigdata/namespace/kb/sparql",
                              cim_profile='rc4_2021', iec61970_301=7)
blazegraph = BlazegraphConnection(params)

Create CIM EquipmentContainer object:

In [ ]:
feeder_mrid = "49AD8E07-3BF9-A4E2-CB8F-C3722F837B62"  # 13 bus
feeder = cim.Feeder(mRID=feeder_mrid)

In [ ]:
network = FeederModel(connection=graphdb, container=feeder, distributed=False)

In [ ]:
network = FeederModel(connection=blazegraph, container=feeder, distributed=False)


## Traversing the Property Graph

![sc-line-segment](./images/ac_line_segment.png)

In [ ]:
network.pprint(cim.ACLineSegment)

## Example 1: Expand the Property Graph by One Edge

In [ ]:
network.get_all_edges(cim.ACLineSegment)

![get-all-edges](./images/get_all_line_edge.png)

In [ ]:
network.pprint(cim.ACLineSegment)

## Example 2: Expand CIM-Graph to Find Bus and Phase of Each Line

In [ ]:
network.get_all_edges(cim.ACLineSegment)
network.get_all_edges(cim.ACLineSegmentPhase)
network.get_all_edges(cim.Terminal)

![traverse-graph](./images/traverse_graph.png)

In [ ]:
network.pprint(cim.ACLineSegmentPhase)

In [ ]:
for line in network.graph[cim.ACLineSegment].values():
    print('\n line mrid: ',line.mRID)
    print('line name:', line.name)
    print('bus 1: ', line.Terminals[0].ConnectivityNode.name)
    print('bus 2: ', line.Terminals[1].ConnectivityNode.name)
    
    for line_phs in line.ACLineSegmentPhases:
        print('phase:', line_phs.phase, ', sequence:', line_phs.sequenceNumber)

## Example 3: Get All Measurements
All SCADA points are associated in memory with the correct power system objects

In [ ]:
network.get_all_edges(cim.Analog)
network.get_all_edges(cim.Discrete)

In [ ]:
for line in network.graph[cim.ACLineSegment].values():
    for meas in line.Measurements:
        print('Measurement: ', meas.name,  ', type:', meas.measurementType, ', phases:', meas.phases)

## Example 4: Get all line impedances and physical asset info

![LineModel.png](attachment:LineModel.png)

In [ ]:
network.get_all_edges(cim.ACLineSegment)
network.get_all_edges(cim.ACLineSegmentPhase)
network.get_all_edges(cim.PerLengthPhaseImpedance)
network.get_all_edges(cim.PhaseImpedanceData)
network.get_all_edges(cim.WireSpacingInfo)
network.get_all_edges(cim.WirePosition)
network.get_all_edges(cim.OverheadWireInfo)
network.get_all_edges(cim.ConcentricNeutralCableInfo)
network.get_all_edges(cim.TapeShieldCableInfo)
network.get_all_edges(cim.Terminal)

### Example 4.1: Parse by PSR:

In [ ]:
for line in network.graph[cim.ACLineSegment].values():
    print('\n line mrid: ', line.mRID)
    print('line name:', line.name)

    for line_phs in line.ACLineSegmentPhases:
        print('phase ', line_phs.phase, ': ', line_phs.mRID)
        if line_phs.WireInfo is not None:
            print('type: ', line_phs.WireInfo.__class__.__name__)
            print('gmr: ', line_phs.WireInfo.gmr)
            print('insulated: ', line_phs.WireInfo.insulated)

    if line.WireSpacingInfo is not None:
        for position in line.WireSpacingInfo.WirePositions:
            print('seq:', position.sequenceNumber, ' x:', position.xCoord, ' y:', position.yCoord)    

    if line.PerLengthImpedance is not None:
        for data in line.PerLengthImpedance.PhaseImpedanceData:
            print('row:', data.row, 'col:', data.column, 'r:', data.r, 'x:', data.x, 'b:', data.b)

### Example 4.2: Parse by Asset

In [ ]:
for impedance in network.graph[cim.PerLengthPhaseImpedance].values():
    print('\n name:', impedance.name)
    for data in impedance.PhaseImpedanceData:
            print('row:', data.row, 'col:', data.column, 'r:', data.r, 'x:', data.x, 'b:', data.b)
    for line in impedance.ACLineSegments:
        node1 = line.Terminals[0].ConnectivityNode
        node2 = line.Terminals[1].ConnectivityNode
        print('Line: ', line.name)
        print('Buses:', node1.name, node2.name)

In [ ]:
for line in network.graph[cim.ACLineSegment].values():
    for meas in line.Measurements:
        print('Measurement: ', meas.name,  ', type:', meas.measurementType, ', phases:', meas.phases)

In [ ]:
metrics = {}

if cim.Substation in network.graph:
    for substation in network.graph[cim.Substation].values():
        metrics['substation_name'] = substation.name
else:
    metrics['substation_name'] = None

metrics["lv_len_km"] = 0
metrics["mv_len_km"] = 0
metrics["mv_3ph_len_km"] = 0
metrics["mv_2ph_len_km"] = 0
metrics["mv_1ph_len_km"] = 0

for base_voltage in network.graph[cim.BaseVoltage].values():
    if float(base_voltage.nominalVoltage) > 1000:
    
        for equipment in base_voltage.ConductingEquipment:
            if equipment.__class__.__name__ == 'ACLineSegment':
                metrics["mv_len_km"] = metrics["mv_len_km"] + float(equipment.length)/1000
                phases = []
                for phase in equipment.ACLineSegmentPhases:
                    phases.append(str(phase.phase).split('SinglePhaseKind.')[1])
                    if 'N' in phases:
                        num_phases = len(phases) - 1
                    else:
                        num_phases = len(phases)
                    if 's1' in phases or 's2' in phases:
                        _log.warning(f'Triplex line over 1000 V with mRID {equipment.mRID}')
            

                if num_phases == 3:
                    metrics["mv_3ph_len_km"] = metrics["mv_3ph_len_km"] + float(equipment.length)/1000
                elif num_phases == 2:
                    metrics["mv_2ph_len_km"] = metrics["mv_2ph_len_km"] + float(equipment.length)/1000
                elif num_phases == 1:
                    metrics["mv_1ph_len_km"] = metrics["mv_1ph_len_km"] + float(equipment.length)/1000

metrics


In [ ]:
metrics['num_regulators'] = len(network.graph[cim.RatioTapChanger].keys())

metrics['num_regulator_banks'] = 0
metrics['num_distribution_transformers'] = 0
metrics['num_3ph_transformers'] = 0
metrics['num_1ph_transformers'] = 0

for transformer in network.graph[cim.PowerTransformer].values():
    # for end in transformer.PowerTransformerEnd # skipping LTC's
    tap = False
    for tank in transformer.TransformerTanks:
        for end in tank.TransformerTankEnds:
            if end.RatioTapChanger is not None:
                tap = True
    if tap == True:
        metrics['num_regulator_banks'] += 1
    else: 
        metrics['num_distribution_transformers'] += 1

    if transformer.TransformerTanks == []:
        metrics['num_3ph_transformers'] += 1
    elif len(transformer.TransformerTanks) == 1:
        metrics['num_1ph_transformers'] += 1 #TODO: use terminals / windings count

    

metrics


In [ ]:
metrics['num_fuses'] = count_device(network, cim.Fuse)
metrics['num_sectionalizers'] = count_device(network, cim.Sectionaliser)
metrics['num_reclosers'] = count_device(network, cim.Recloser)
metrics['num_breakers'] = count_device(network, cim.Breaker)
metrics['num_switches'] = count_device(network, cim.LoadBreakSwitch) + count_device(network, cim.Switch)


metrics


In [ ]:
# get ditto capacitor info

for element in network.graph[cim.LinearShuntCompensator].values():
    susceptance = float(element.bPerSection)
    conductance = float(element.gPerSection)
    # resistance = 1 / conductance #not in CIM
    susceptance0 = float(element.b0PerSection)
    conductance0 = float(element.g0PerSection)
    # resistance0 = 1 / conductance0
    connecting_element = element.Terminals[0].ConnectivityNode

    if element.RegulatingControl is not None:
        high = element.RegulatingControl #TODO should  be vmax in opendss -- figure out Tom's converter 
        low = element.RegulatingControl #TODO should  be vmax in opendss -- figure out Tom's converter 
        mode = element.RegulatingControl.mode #enum
    delay = float(element.aVRDelay)
    nominal_voltage = float(element.nomU)

    positions = element.Location.PositionPoints

    measuring_element = None #TODO associate with OpenDSS line object meas
    
    pt_ratio = None #missing?
    pt_phase = None
    ct_ratio = None #missing?
    

    connection_type = element.phaseConnection

    container = element.EquipmentContainer
    if container.__class__.__name__ == "Feeder":
        feeder_name = container.name
        substation_name = container.NormalEnergizingSubstation.name
        is_substation = False
    elif container.__class__.__name__ == "Substation":
        feeder_name = None
        substation_name = container.name
        is_substation = True
    else:
        feeder_name = None
        try:
            substation_name = container.Substation.name
        except:
            _log.warning("could not identify feeder or substation")

    for phase in element.ShuntCompensatorPhase:
        switch = phase.sections #TODO: figure out difference between switch and sections
        sections = phase.sections 
        normalsections = phase.normalSections
        var = nominal_voltage * nominal_voltage * float(phase.bPerSection)

        

